<a href="https://colab.research.google.com/github/Tamanna-op/flyrank-ml-internship-task/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamanna-op/flyrank-ml-internship-task/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

### Lane: Refresh / Content Opportunity Scoring

#### ML task type: Ranking / scoring

The goal is to rank pages by how strongly the available evidence suggests that they deserve human review.

The practical problem is not simply to classify every page as "needs a refresh" or "does not need a refresh." A content team has limited review capacity, so the more useful output is an ordered list of pages that should be considered first.

The eventual system should use observable page-level signals to produce a review-priority score or ranking. The ranking can then support actions such as refresh, expansion, protection, pruning/consolidation review, or monitoring.

This is a decision-support task: the model recommends which pages to investigate first, while a human reviewer makes the final content decision.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("\nExample page-level records:")
display(
    df[
        [
            "content_id",
            "client_id",
            "impressions_90d",
            "avg_position",
            "ctr",
            "content_age_days",
            "days_since_last_update",
            "trend_direction"
        ]
    ].head()
)

Dataset shape: (30000, 44)

Example page-level records:


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,187,20,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,445,25,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,141,20,down
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,463,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,263,14,down


## 2. Target or proxy

For the starter implementation, I will use is_declining_label as a proxy target:

is_declining_label = 1 when trend_direction == "down", otherwise 0.

This proxy represents whether a page is currently showing a declining trend in the available observation window.

I am using it because it is already available in the starter dataset and allows me to test the ML workflow. However, it is not the ideal target for the final project. A stronger target would be future-looking, for example:

features from a prior window → decline or recovery during a later window

That would better match the real decision point because the model would use information available before the decision and predict an outcome that happens afterward.

Therefore, I will treat is_declining_label as a starter proxy rather than claiming that it represents whether a refresh will succeed.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the starter proxy target.

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Target/proxy distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget/proxy proportions:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Target/proxy distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target/proxy proportions:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

The primary success metric will be Precision@K, with Precision@50 as the initial K.

Precision@50 measures how many of the top 50 ranked pages match the starter proxy target.

This metric matches the real decision because a content team has limited review capacity. If the team can review approximately 50 pages, I care more about whether the top of the ranking contains useful candidates than about getting every page classified correctly.

I will also consider recall and average precision as supporting metrics, but Precision@50 will be the main metric for the initial ranking experiment.

For the eventual project, the metric should be tied to the actual review capacity and decision cost rather than chosen as a generic ML metric.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Initial review capacity used for evaluation.

K = 50

print("Evaluation metric: Precision@K")
print("K =", K)
print("Interpretation: fraction of the top", K, "ranked pages that match the target/proxy.")

Evaluation metric: Precision@K
K = 50
Interpretation: fraction of the top 50 ranked pages that match the target/proxy.


## 4. The unit of analysis, as a real dataframe
The unit of analysis is one page/content item.

Each row represents one page and contains its observed search, engagement, and content-related signals.

For this task, examples of candidate input features include:

impressions_90d
avg_position
ctr
content_age_days
days_since_last_update
word_count
engagement-related signals

The target/proxy is stored separately as is_declining_label.

The page identifier is used to identify the row, not as a predictive feature. Similarly, client_id can be useful for grouping and validation, but the identifier itself should not be treated as a meaningful numeric feature.


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis directly.
# One row = one page/content item.

page_view = df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "word_count",
        "trend_direction",
        "is_declining_label"
    ]
].head(10)

display(page_view)

,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,187,20,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,445,25,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,141,20,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,463,22,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,263,14,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,0.03,147,20,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,7.0,0.00,90,20,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,21.2,0.06,445,22,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,46.0,0.09,90,20,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,4.9,0.16,257,104,NaN,down,1


## 5. Why ML beats a fixed rule here

A fixed rule is useful as a transparent baseline, but it requires manually choosing thresholds and combinations of signals.

For example, a rule could prioritize pages that are both stale and highly visible. However, this assumes in advance that those thresholds and conditions are the best way to identify useful review candidates.

ML can test combinations of multiple observable signals and learn patterns from the available examples instead of relying entirely on manually chosen thresholds.

My Week 2 experiment provides an initial reason to investigate this further. In client-holdout validation, the hand rule achieved 0.420 Precision@50, while the depth-4 decision tree achieved 0.620 Precision@50 on unseen clients in that experiment.

This does not prove that ML will always outperform a fixed rule. It suggests that a learned ranking is worth testing more carefully against a transparent baseline using appropriate validation.

The fixed rule should therefore remain as a baseline that the ML approach has to earn the right to beat.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.